In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import os, sys
import pandas as pd


current_dir = os.getcwd()
parent_dir = os.path.dirname(f"{"\\".join(current_dir.split("\\"))}")
sys.path.append(parent_dir)

from constants.maps import ELV_ACCOUNT_TYPE_MAP

from utils.clickup import fix_clickup_date
from utils.collector import (
    get_all_elv_organizations,
    get_all_elv_plans,
    get_all_cu_clients,
    get_all_cu_plans,
)

In [ ]:
''' collect cached data '''
elv_org_df = get_all_elv_organizations()
elv_plan_df = get_all_elv_plans(oids=elv_org_df['organization_id'].unique())
cu_client_df = get_all_cu_clients()
cu_plan_df = get_all_cu_plans()


for col in ['date_plan_start', 'date_plan_end']:
    cu_plan_df[col] = cu_plan_df[col].apply(fix_clickup_date)


In [ ]:
''' merge elv orgs with elv plans '''
# easy one liner
elv_org_plans = pd.merge(left=elv_plan_df, right=elv_org_df, on='organization_id', how='left')

In [ ]:
''' merge cu clients with cu plan cards '''
# not as easy cuz dirty relation fields

relation_fields = [c for c in cu_plan_df.columns if 'client_' in c]

cu_plan_df[relation_fields] = cu_plan_df[relation_fields].apply(
    lambda col: col.apply(lambda x: x if isinstance(x, list) else [])
)
# combine all 'client_x' relation field values into one column
cu_plan_df['client_ids'] = cu_plan_df.apply(lambda row: set(sum(row[relation_fields].values, [])), axis=1)

# all plan cards SHOULD have one client linked, so raise if more than one is linked
# the number of plans with no client linked will get printed but not raised
cu_plan_df['len_client_ids'] = cu_plan_df['client_ids'].apply(lambda x: len(x))
for num_clients, num_plans in cu_plan_df['len_client_ids'].value_counts().to_dict().items():
    # 1 client linked
    if num_clients == 1:
        # this is perfect
        continue

    # 0 clients linked
    if num_clients == 0:
        # this usually happens when clients are onboarding
        print(f"{num_plans:>5} plans don't have a client linked!")
        continue

    # 2+ clients linked, this shouln't happen...
    print(f"{num_plans:>5} plans have {num_clients} linked!")
    print(f"THESE NEED TO BE FIXED BEFORE CONINUING!!!")
    raise ValueError("More than one client linked!")

# extract client id from list
cu_plan_df['client_id'] = cu_plan_df['client_ids'].apply(lambda x: list(x)[0] if len(x) > 0 else None)

# drop columns used to extract client id
cu_plan_df = cu_plan_df.drop(columns=['client_ids', 'len_client_ids'])

cu_client_plan_df = pd.merge(left=cu_plan_df, right=cu_client_df, on='client_id', how='left')

In [ ]:
from datetime import datetime as dt
from dateutil.relativedelta import relativedelta

elv_plans = elv_org_plans.copy()

elv_plans['cu_plan_id'] = [[] for x in range(len(elv_plans))]
cu_client_plan_df['elv_plan_id'] = [[] for x in range(len(cu_client_plan_df))]


for rmrcode, org_plans in elv_plans.groupby("rmrcode"):
    clickup_plans = cu_client_plan_df[cu_client_plan_df['rmrcode'] == rmrcode]

    for index, row in org_plans.iterrows():

        if row['account_type.account_type'] == "HSA":
            cu_plan_search = clickup_plans[
                clickup_plans['cu_account_type'] == "HSA"
            ]

        else:
            cu_plan_search = clickup_plans[
                (clickup_plans['cu_account_type'] == ELV_ACCOUNT_TYPE_MAP.get(row['account_type.account_type'])) &
                (clickup_plans['date_plan_start'] > row['plan_year.valid_from'] - relativedelta(days=1)) &
                (clickup_plans['date_plan_start'] < row['plan_year.valid_from'] + relativedelta(days=1))
            ]

        if not cu_plan_search.empty:
            elv_plans.iloc[index]['cu_plan_id'] = elv_plans.iloc[index]['cu_plan_id'].extend(cu_plan_search['cu_plan_id'].to_list())



In [ ]:
elv_plans[elv_plans['cu_plan_id'].apply(lambda x: len(x) > 1)]

In [ ]:
print(cu_plans_copy.shape)

In [ ]:
cu_plans_copy = cu_client_plan_df.copy()

cu_plans_copy = cu_plans_copy.drop(columns=[
    "date_created",
    "date_updated",
    "date_closed",
    "date_done",
    "start_date",
    "due_date"
])


for col in [c for c in cu_plans_copy.columns if "date" in c ]:
    print(col)





In [ ]:
elv_plans_w_multiple_cu_cards = elv_plans[elv_plans['cu_plan_id'].apply(lambda x: len(x) == 2)]

for index, row in elv_plans_w_multiple_cu_cards.iterrows():
    print(f"{row.get('elv_plan_id'):<6} {row.get('account_type.account_type'):<12} "
          f"{dt.strftime(row.get('plan_year.valid_from'), '%m/%d/%Y'):<12} {row.get('plan_code'):<20}")
    
    clickup_plans = cu_plans_copy[cu_plans_copy['cu_plan_id'].isin(row.get("cu_plan_id"))]
    
    if len(clickup_plans) != 2:
        print(f"\tWarning: Expected 2 ClickUp plans, found {len(clickup_plans)}")
        continue

    plan1, plan2 = clickup_plans.iloc[0], clickup_plans.iloc[1]
    
    for col in clickup_plans.columns:
        val1, val2 = plan1[col], plan2[col]
        
        # Convert everything to string for comparison
        str1, str2 = str(val1), str(val2)
        
        # Skip if both are "nan" (string representation of NaN)
        if str1 == "nan" and str2 == "nan":
            continue
            
        # Compare string representations
        if str1 != str2:
            print(f"\t{col}")
            print(f"\t\t{str1} != {str2}")

    break